In [4]:
# import cv2
# import numpy as np

# # Parameters
# min_contour_width = 40
# min_contour_height = 40
# offset = 10
# line_height = 550
# vehicles = 0

# matches = []

# # Correct centroid function
# def get_centroid(x, y, w, h):
#     cx = x + int(w / 2)
#     cy = y + int(h / 2)
#     return cx, cy

# cap = cv2.VideoCapture('Video.mp4')
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

# ret, frame1 = cap.read()
# ret, frame2 = cap.read()

# while ret:
#     diff = cv2.absdiff(frame1, frame2)
#     gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
#     blur = cv2.GaussianBlur(gray, (5, 5), 0)

#     _, thresh = cv2.threshold(blur, 20, 255, cv2.THRESH_BINARY)
#     dilated = cv2.dilate(thresh, np.ones((3, 3), np.uint8), iterations=2)

#     kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
#     closing = cv2.morphologyEx(dilated, cv2.MORPH_CLOSE, kernel)

#     contours, _ = cv2.findContours(
#         closing, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
#     )

#     cv2.line(frame1, (0, line_height), (1200, line_height), (0, 255, 0), 2)

#     for c in contours:
#         x, y, w, h = cv2.boundingRect(c)

#         if w < min_contour_width or h < min_contour_height:
#             continue

#         centroid = get_centroid(x, y, w, h)
#         matches.append(centroid)

#         cv2.rectangle(frame1, (x, y), (x + w, y + h), (255, 0, 0), 2)
#         cv2.circle(frame1, centroid, 5, (0, 255, 0), -1)

#     # Count vehicles
#     for (x, y) in matches:
#         if line_height - offset < y < line_height + offset:
#             vehicles += 1
#             matches.remove((x, y))

#     cv2.putText(
#         frame1,
#         f"Total Vehicle Detected: {vehicles}",
#         (10, 50),
#         cv2.FONT_HERSHEY_SIMPLEX,
#         1,
#         (0, 255, 0),
#         2
#     )

#     cv2.imshow("Vehicle Detection", frame1)

#     if cv2.waitKey(1) == 27:
#         break

#     frame1 = frame2
#     ret, frame2 = cap.read()

# cap.release()
# cv2.destroyAllWindows()


In [5]:
import cv2
import numpy as np

min_contour_width = 40
min_contour_height = 40
offset = 10
line_height = 550
vehicles = 0

# Dictionary to track vehicles
vehicle_tracks = {}
vehicle_id = 1

def get_centroid(x, y, w, h):
    cx = x + int(w/2)
    cy = y + int(h/2)
    return cx, cy

cap = cv2.VideoCapture("Video.mp4")
ret, frame1 = cap.read()
ret, frame2 = cap.read()

while ret:
    diff = cv2.absdiff(frame1, frame2)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    _, thresh = cv2.threshold(blur, 20, 255, cv2.THRESH_BINARY)
    dilated = cv2.dilate(thresh, np.ones((3,3), np.uint8), iterations=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2,2))
    closing = cv2.morphologyEx(dilated, cv2.MORPH_CLOSE, kernel)
    
    contours, _ = cv2.findContours(closing, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    current_centroids = []

    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w < min_contour_width or h < min_contour_height:
            continue

        cx, cy = get_centroid(x, y, w, h)
        current_centroids.append((cx, cy))
        cv2.rectangle(frame1, (x,y), (x+w, y+h), (255,0,0), 2)
        cv2.circle(frame1, (cx,cy), 5, (0,255,0), -1)

    # --- Vehicle Tracking ---
    new_tracks = {}
    for cx, cy in current_centroids:
        matched = False
        for vid, data in vehicle_tracks.items():
            prev_cx, prev_cy, counted = data
            if abs(cx - prev_cx) <= 50 and abs(cy - prev_cy) <= 50:
                # same vehicle
                new_tracks[vid] = [cx, cy, counted]
                matched = True
                break
        if not matched:
            # new vehicle
            new_tracks[vehicle_id] = [cx, cy, False]
            vehicle_id += 1

    # Check if any vehicle crossed the line
    for vid, data in new_tracks.items():
        cx, cy, counted = data
        if not counted and line_height - offset < cy < line_height + offset:
            vehicles += 1
            new_tracks[vid][2] = True  # mark as counted

    vehicle_tracks = new_tracks

    cv2.line(frame1, (0, line_height), (1200, line_height), (0,255,0), 2)
    cv2.putText(frame1, f"Vehicles: {vehicles}", (10,50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    cv2.imshow("Vehicle Detection", frame1)

    key = cv2.waitKey(1) & 0xFF
    if key == 27 or cv2.getWindowProperty("Vehicle Detection", cv2.WND_PROP_VISIBLE) < 1:
        break

    frame1 = frame2
    ret, frame2 = cap.read()

cap.release()
cv2.destroyAllWindows()
